In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polatory as p
import polatory.three as p3

def variog(m: p3.Model, diff: np.ndarray) -> float:
    v = m.nugget
    for rbf in m.rbfs:
        v += rbf.evaluate(np.zeros(3)) - rbf.evaluate(diff)
    return v

In [2]:
data = pd.read_csv("data/grade.csv")
points = data[["X", "Y", "Z"]].values
hole_ids = data["Hole ID"].values
values = data["Grade Values"].values
hids = np.unique(hole_ids)
hole_indices = np.array([np.where(hids == hid)[0] for hid in hole_ids])
set_ids = hole_indices % 4

In [3]:

directions = np.array(
    [
        [0.0,0.0,1.0],
        [0.10607892,0.32647735,0.9392336],
        [-0.27771825,0.20177411,0.9392336],
        [-0.18759248,0.57735026,0.79465449],
        [-0.55543649,0.40354821,0.72707576],
        [-0.44935754,0.73002559,0.51491791],
        [-0.72360682,0.52573109,0.44721359],
        [0.21215785,0.6529547,0.72707576],
        [-0.065560378,0.85472882,0.51491791],
        [0.2763932,0.85065079,0.44721359],
        [-0.27771825,-0.20177411,0.9392336],
        [-0.60706198,0.0,0.79465449],
        [-0.55543649,-0.40354821,0.72707576],
        [-0.83315468,-0.20177411,0.51491791],
        [-0.72360682,-0.52573109,0.44721359],
        [-0.83315468,0.20177411,0.51491791],
        [0.10607892,-0.32647735,0.9392336],
        [-0.18759248,-0.57735026,0.79465449],
        [0.21215785,-0.6529547,0.72707576],
        [-0.065560378,-0.85472882,0.51491791],
        [0.2763932,-0.85065079,0.44721359],
        [-0.44935754,-0.73002559,0.51491791],
        [0.34327862,0.0,0.9392336],
        [0.49112347,-0.3568221,0.79465449],
        [0.68655723,0.0,0.72707576],
        [0.79263616,-0.32647735,0.51491791],
        [0.89442718,0.0,0.44721359],
        [0.55543649,-0.6529547,0.51491791],
        [0.49112347,0.3568221,0.79465449],
        [0.55543649,0.6529547,0.51491791],
        [0.79263616,0.32647735,0.51491791],
        [0.10607892,0.97943211,0.17163931],
        [-0.30353099,0.93417233,0.18759248],
        [-0.66151541,0.73002559,0.17163931],
        [-0.89871508,0.40354821,0.17163931],
        [-0.98224694,0.0,0.18759248],
        [-0.89871508,-0.40354821,0.17163931],
        [-0.66151541,-0.73002559,0.17163931],
        [-0.30353099,-0.93417233,0.18759248],
        [0.10607892,-0.97943211,0.17163931],
        [0.48987609,-0.85472882,0.17163931],
        [0.79465449,-0.57735026,0.18759248],
        [0.96427548,-0.20177411,0.17163931],
        [0.96427548,0.20177411,0.17163931],
        [0.79465449,0.57735026,0.18759248],
        [0.48987609,0.85472882,0.17163931],
    ]
)

variogs = p3.VariogramCalculator(points, values, bin_width=5.0, num_bins=200,
                                 angle_tolerance=np.radians(22.5), directions=directions).variograms

In [4]:
cov1 = p3.CovSpherical([0.6, 50.0])
covs = [cov1]
m = p3.Model(covs, poly_degree=0)
res = p3.cross_validation(m, points, values, set_ids, 1e-3)
print("Residual before fitting:", np.linalg.norm(res))

best_fit = None
best_cost = float("inf")
for _ in range(30):
    weight = p.WeightFunction(exp_num_pairs=1.0, exp_distance=-2.0)
    fit = p.VarioFitting(variogs, m, weight)
    if fit.final_cost < best_cost:
        best_fit = fit
        best_cost = fit.final_cost

m = best_fit.model
res = p3.cross_validation(m, points, values, set_ids, 1e-3)
print("Residual after fitting:", np.linalg.norm(res))

/var/folders/s8/spt9kxws33n8ltk3w4zrlds00000gn/T/ipykernel_73917/3175554845.py:4: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  res = p3.cross_validation(m, points, values, set_ids, 1e-3)


   level       n_domains        n_points
Residual before fitting: 235.08997810044252
       1             128           46262
       0               1            1027
iter         rel_res
   0    1.000000e+00
   1    2.759187e-01
   2    8.331133e-02
   3    3.528368e-02
   4    1.579191e-02
   5    8.074986e-03
   6    3.612810e-03
   7    1.547074e-03
   8    5.353929e-04
   9    2.153952e-04
  10    1.006288e-04
  11    4.873986e-05
  12    3.203997e-05
  13    1.788404e-05
Achieved absolute residual: 0.000728006
   level       n_domains        n_points
       1             128           46202
       0               1            1020
iter         rel_res
   0    1.000000e+00
   1    5.522728e-01
   2    1.500074e-01
   3    6.089416e-02
   4    3.197237e-02
   5    2.041882e-02
   6    1.229898e-02
   7    8.008380e-03
   8    5.370030e-03
   9    2.583393e-03
  10    1.142107e-03
  11    5.491092e-04
  12    2.510776e-04
  13    1.447050e-04
  14    9.432397e-05
  15    4.842997e-0

/var/folders/s8/spt9kxws33n8ltk3w4zrlds00000gn/T/ipykernel_73917/3175554845.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  res = p3.cross_validation(m, points, values, set_ids, 1e-3)


In [5]:
ea = np.degrees(best_fit.euler_angles)
pitch, dip, dip_az = ea
if dip < -90.0:
    dip += 180.0
    pitch = 180.0 - pitch
elif dip < 0.0:
    dip = -dip
    dip_az += 180.0
elif dip > 90.0:
    dip = 180.0 - dip
    dip_az += 180.0
    pitch = 180.0 - pitch
if dip_az < 0.0:
    dip_az += 360.0
elif dip_az > 360.0:
    dip_az -= 360.0
print("         dip      dip az       pitch")
print(f"  {dip:10.4f}  {dip_az:10.4f}  {pitch:10.4f}")

print("       psill         maj         mid         min")
nug = best_fit.parameters[0]
print(f"  {nug:10.4f}")
for i in range(len(covs)):
    psill = best_fit.parameters[1 + 2 * i]
    r = best_fit.parameters[1 + 2 * i + 1]
    maj = r / best_fit.scale(i)[0]
    mid = r / best_fit.scale(i)[1]
    min = r / best_fit.scale(i)[2]
    print(f"  {psill:10.4f}  {maj:10.4f}  {mid:10.4f}  {min:10.4f}")

         dip      dip az       pitch
     82.6662    169.1461     69.7978
       psill         maj         mid         min
      0.0218
      0.5861     91.3026     34.4065      9.8783


In [ ]:

fig, ax = plt.subplots(nrows=12, ncols=4, figsize=(16, 32))
fig.tight_layout()
for i in range(12):
    for j in range(4):
        if 4 * i + j >= len(variogs):
            break
        v = variogs[4 * i + j]
        dir = v.direction
        az = np.degrees(np.arctan2(dir[0], dir[1]))
        if az < 0.0:
            az += 360.0
        incl = np.degrees(np.arccos(dir[2]))
        xs = np.linspace(0.0, 1000.0, 1000)
        ys = np.array([variog(m, x * dir) for x in xs])
        ax[i, j].plot(v.bin_distance, v.bin_gamma, "o")
        ax[i, j].plot(xs, ys, "-")
        ax[i, j].set_title(f"Az: {az:.0f}°, Incl: {incl:.0f}°")
        ax[i, j].set_xlim(0.0, 200.0)
        ax[i, j].set_ylim(0.0, 1.2)
        ax[i, j].margins(0.0)

In [ ]:
inter = p3.Interpolant(m)
inter.fit(points, values, absolute_tolerance=1e-4)

In [ ]:
fn = p.RbfFieldFunction(inter)
bbox = p3.Bbox.from_points(points)
iso = p.Isosurface(bbox, resolution=10.0)
surf = iso.generate(fn, isovalue=0.09)
surf.export_obj("data/au_0.09.obj")
surf = iso.generate(fn, isovalue=0.30)
surf.export_obj("data/au_0.30.obj")
surf = iso.generate(fn, isovalue=0.63)
surf.export_obj("data/au_0.63.obj")